# FitLoop - Data cleaning

`users.csv` and `marketing_spend.csv` are a raw production extract . Before measuring CAC, retention or LTV we make sure the data can be trusted. not cleaned data would give wrong unit economics and a wrong recommendation to the board.

In [27]:
import pandas as pd
import numpy as np

## Load raw data

In [28]:
mark_raw=pd.read_csv("../data/marketing_spend.csv")
display(mark_raw.head(3))
user_raw=pd.read_csv("../data/users.csv")
display(user_raw.head(3))

,month,channel,spend_eur
0,2025-01-01,Meta Ads,3750.0
1,2025-01-01,Google Ads,3000.0
2,2025-01-01,TikTok Ads,0.0


,user_id,signup_date,channel,country,platform,activated_d7,trial_start_date,subscription_start_date,plan,subscription_end_date
0,U101247,2025-01-20,Organic,FR,iOS,0,NaN,NaN,NaN,NaN
1,U119636,2025-08-19,TikTok Ads,FR,Android,1,2025-08-20,NaN,NaN,NaN
2,U105849,2025-03-16,Organic,BE,Android,1,NaN,NaN,NaN,NaN


In [29]:
print(user_raw.shape,mark_raw.shape)

(70851, 10) (71, 3)


In [30]:
mark=mark_raw.copy()
user=user_raw.copy()

## Fixing formats

In [31]:
print(mark.dtypes)
print(user.dtypes)

month            str
channel          str
spend_eur    float64
dtype: object
user_id                      str
signup_date                  str
channel                      str
country                      str
platform                     str
activated_d7               int64
trial_start_date             str
subscription_start_date      str
plan                         str
subscription_end_date        str
dtype: object


**Dates**: the date columns are text (str), so we convert them to real dates. A value that can’t be read becomes empty (NaT)

**Numbers**: `spend_eur` and `activated_d7` must be numeric. Anything that isn’t a number becomes NaN.

**Categories**: `channel`, `country`, platform and plan are labels with a few distinct values. We strip stray spaces and convert them to category.

#### 0. count filled values before(to see if the conversion loses anything)


In [32]:
na_user, na_mark = user.isna().sum(), mark.isna().sum()

#### 1. converting dates columns with string type to datetime


In [33]:
date_cols = ["signup_date", "trial_start_date", "subscription_start_date", "subscription_end_date"]

for c in date_cols:
    user[c] = pd.to_datetime(user[c], format="ISO8601", errors="coerce").fillna(
              pd.to_datetime(user[c], format="%d/%m/%Y", errors="coerce"))

mark["month"] = pd.to_datetime(mark["month"], format="ISO8601", errors="coerce")

#### 2. converting numeric columns with string type to numeric type


In [34]:
mark["spend_eur"] = pd.to_numeric(mark["spend_eur"], errors="coerce")
user["activated_d7"] = pd.to_numeric(user["activated_d7"], errors="coerce")

#### 3. converting columns with few distinct values to categorical columns (strip spaces, then category)


In [35]:
user["channel"] = user["channel"].str.strip().astype("category")
user["country"] = user["country"].str.strip().astype("category")
user["platform"] = user["platform"].str.strip().astype("category")
user["plan"] = user["plan"].str.strip().astype("category")
mark["channel"] = mark["channel"].str.strip().astype("category")

#### 4. checking if there are any lost columns


In [36]:
print(pd.DataFrame({"dtype": user.dtypes, "missing_added": user.isna().sum() - na_user}))
print(pd.DataFrame({"dtype": mark.dtypes, "missing_added": mark.isna().sum() - na_mark}))

                                  dtype  missing_added
user_id                             str              0
signup_date              datetime64[us]              0
channel                        category              0
country                        category              0
platform                       category              0
activated_d7                      int64              0
trial_start_date         datetime64[us]              0
subscription_start_date  datetime64[us]              0
plan                           category              0
subscription_end_date    datetime64[us]              0
                    dtype  missing_added
month      datetime64[us]              0
channel          category              0
spend_eur         float64              0


## Standardise labels

The same real-world value must be written one way only. If `tiktok` and `TikTok Ads` stay separate, one channel is split in two and CAC, conversion and retention by channel are wrong. The channel names must also match exactly between `users` and `marketing_spend`, so we can join them.


#### 1. Country in `user.csv`

only FR, BE, CH, LU, NAN (no spelling, case or space variants).

In [37]:
user.country.value_counts(dropna=False)

country
FR     48519
BE     10460
CH      6926
LU      3436
NaN     1510
Name: count, dtype: int64

#### 2. Platfrom in `user.csv`

same for platform only iOS, Android (no spelling, case or space variants).

In [38]:
(user.platform.value_counts(dropna=False))


platform
iOS        36037
Android    34814
Name: count, dtype: int64

#### 3. Plan in `user.csv`

same for plan only monthly, annual,NaN.(no spelling, case or space variants).

In [39]:
(user["plan"].value_counts(dropna=False))

plan
NaN        63695
monthly     5404
annual      1752
Name: count, dtype: int64

#### 4. channel in `user.csv` and `marketing_spend.csv`

in `marketing_spend.csv` don't need intervention only Google Ads,
Influencers,
TikTok Ads,
Meta Ads (no spelling, case or space variants).

In [40]:
(mark["channel"].value_counts(dropna=False))

channel
Google Ads     18
Influencers    18
TikTok Ads     18
Meta Ads       17
Name: count, dtype: int64

in `users.csv`  there are 11 labels. The same channels were written several ways: `tiktok` / `TikTok Ads`, `meta_ads` / `Facebook Ads` / `Meta Ads`, and `google_ads` / `Google Ads`.

In [41]:
(user["channel"].value_counts(dropna=False))

channel
TikTok Ads      17557
Meta Ads        16619
Organic         10166
Google Ads       8370
Influencers      7760
tiktok           3055
Referral         2947
meta_ads         1481
Facebook Ads     1436
google_ads       1415
internal           45
Name: count, dtype: int64

In [42]:
channel_map = {
    "tiktok":       "TikTok Ads",
    "meta_ads":     "Meta Ads",
    "Facebook Ads": "Meta Ads",
    "google_ads":   "Google Ads",
}

user["channel"] = user["channel"].replace(channel_map).cat.remove_unused_categories()

print(user["channel"].value_counts(dropna=False))

channel
TikTok Ads     20612
Meta Ads       19536
Organic        10166
Google Ads      9785
Influencers     7760
Referral        2947
internal          45
Name: count, dtype: int64


after correction we have only 7 labels: TikTok Ads (20,612), Meta Ads (19,536), Organic (10,166), Google Ads (9,785), Influencers (7,760), Referral (2,947) and internal (45).
Total rows are unchanged  as we only renamed. Meta Ads and TikTok Ads are the two largest channels, so the split had hidden their real size.

## Dealing with duplicates

In [43]:
print("rows before:", len(user))
print("exact duplicate rows:", user.duplicated().sum())

user = user.drop_duplicates(keep="first")
print("rows after:", len(user))

rows before: 70851


exact duplicate rows: 562
rows after: 70289


In [44]:
print("rows before:", len(mark))
print("duplicate month+channel:", mark.duplicated(subset=["month", "channel"]).sum())

mark = mark.drop_duplicates(subset=["month", "channel"], keep="first")
print("rows after:", len(mark))

rows before: 71
duplicate month+channel: 0
rows after: 71


## Dealing with impossible values 

Events must follow a logical order and stay inside the observation window (1 Jan 2025 – 30 Jun 2026). Impossible rows give negative lifetimes or users who exist after the extraction date. \
**Rules applied.** Signup outside the window; trial before signup; payment before trial start; subscription end before start; any date after 30 Jun 2026.


In [45]:
start = pd.Timestamp("2025-01-01")
extraction = pd.Timestamp("2026-06-30")

rules = {
    "signup outside Jan 2025 - Jun 2026": (user["signup_date"] < start) | (user["signup_date"] > extraction),
    "trial before signup":                user["trial_start_date"] < user["signup_date"],
    "paid before trial start":            user["subscription_start_date"] < user["trial_start_date"],
    "end before paid start":              user["subscription_end_date"] < user["subscription_start_date"],
    "paid or trial date after extraction": (user["subscription_start_date"] > extraction)
                                          | (user["trial_start_date"] > extraction),
    "end date after extraction":          user["subscription_end_date"] > extraction,
}

for name, m in rules.items():
    print(f"{name}: {m.sum()} rows")

bad = pd.concat(rules.values(), axis=1).any(axis=1)
print("rows to remove (a row can break several rules):", bad.sum())
display(user[bad])
print("rows before:", len(user))
user = user[~bad]
print("rows after:", len(user))


signup outside Jan 2025 - Jun 2026: 7 rows
trial before signup: 0 rows
paid before trial start: 0 rows
end before paid start: 0 rows
paid or trial date after extraction: 0 rows
end date after extraction: 0 rows
rows to remove (a row can break several rules): 7


,user_id,signup_date,channel,country,platform,activated_d7,trial_start_date,subscription_start_date,plan,subscription_end_date
6957,U152193,1900-01-01,Organic,FR,Android,1,2026-03-06,2026-03-13,monthly,2026-06-12
12663,U136285,1900-01-01,Google Ads,FR,iOS,1,NaT,NaT,NaN,NaT
19783,U101299,2099-12-31,Meta Ads,FR,iOS,0,NaT,NaT,NaN,NaT
20443,U113554,1970-01-01,Referral,BE,iOS,1,2025-06-25,NaT,NaN,NaT
55322,U157099,1900-01-01,TikTok Ads,FR,Android,0,NaT,NaT,NaN,NaT
68559,U104630,1900-01-01,Meta Ads,FR,Android,1,2025-03-12,2025-03-19,monthly,2025-12-17
70190,U107118,2099-12-31,Google Ads,NaN,iOS,0,NaT,NaT,NaN,NaT


rows before: 70289
rows after: 70282


In [46]:
# the brief says paying starts at the end of the 7-day trial
gap = (user["subscription_start_date"] - user["trial_start_date"]).dt.days
print(gap.describe())                                   
print("paid with no trial:", (user["subscription_start_date"].notna() & user["trial_start_date"].isna()).sum())

count    7097.000000
mean        6.955615
std         0.555669
min         0.000000
25%         7.000000
50%         7.000000
75%         7.000000
max         7.000000
dtype: float64
paid with no trial: 0


In [47]:
print("negative spend:", (mark["spend_eur"] < 0).sum())
print("missing spend:", mark["spend_eur"].isna().sum())
print(mark["month"].min(), mark["month"].max())          

negative spend: 0
missing spend: 0
2025-01-01 00:00:00 2026-06-01 00:00:00


**Rules checked and results.**

| Rule | Rows |
|---|---|
| Signup outside Jan 2025 – Jun 2026 | 7 |
| Trial starts before signup | 0 |
| Payment starts before trial start | 0 |
| Subscription ends before it starts | 0 |
| Trial or payment date after 30 Jun 2026 | 0 |
| End date after 30 Jun 2026 | 0 |

*We removed the rows that break at least one rule: **7 rows** . These rows are certainly wrong, so removing them is safe.

**Remark.** The three rules that returned 0 show that the later events (payment, end of subscription) are consistent. The errors sit only in the signup and trial dates, which suggests a problem in how the account creation date was recorded, not in the billing data. An empty `subscription_end_date` is not an error: it can means that  the user is still subscribed at extraction.

## Dealing with missing values


To keep every real user in your totals and protect metrics like CAC, conversion, and cohort sizes, we avoid dropping rows with empty fields. 
Not all empty cells are errors:
- Empty trial or subscription dates can  mean that the user never started a trial or paid.
- An empty subscription end date can mean that  the user is still active as of 30 June 2026.
- An empty plan is normal for non-paying users.
Since filling these in would invent data, we leave them untouched.  


for that we will try  Count empties across all columns , separate intentional blanks from genuine missing data, Label missing `channel`, `country`, and `platform` as `Unknown` so users stay in the analysis, and report their share.


In [48]:
print(user.isna().sum())
print(mark.isna().sum())

user_id                        0
signup_date                    0
channel                        0
country                     1494
platform                       0
activated_d7                   0
trial_start_date           50986
subscription_start_date    63185
plan                       63185
subscription_end_date      67669
dtype: int64
month        0
channel      0
spend_eur    0
dtype: int64


In [49]:
user["country"] = user["country"].astype("object").fillna("Unknown").astype("category")
print(user["country"].value_counts(dropna=False))


country
FR         48138
BE         10375
CH          6871
LU          3404
Unknown     1494
Name: count, dtype: int64


In [50]:
print(user["activated_d7"].value_counts(dropna=False))

activated_d7
0    45271
1    25011
Name: count, dtype: int64


In [51]:
plan_no_paid = user["plan"].notna() & user["subscription_start_date"].isna()
paid_no_plan = user["subscription_start_date"].notna() & user["plan"].isna()
end_no_start = user["subscription_end_date"].notna() & user["subscription_start_date"].isna()
paid_no_trial = user["subscription_start_date"].notna() & user["trial_start_date"].isna()

print("plan but never paid:", plan_no_paid.sum())
print("paid but no plan:", paid_no_plan.sum())
print("end date without start date:", end_no_start.sum())
print("paid with no trial (information only):", paid_no_trial.sum())

plan but never paid: 0
paid but no plan: 0
end date without start date: 0
paid with no trial (information only): 0


#### Handle missing values: result

Real missing data Only `country` (1,494 users, about 2%), labelled `Unknown`.

Normal empty values (kept) are  Empty `trial_start_date` (50,986) = no trial; empty `subscription_start_date` (63,185) = never paid; empty `subscription_end_date` (67,669) = still subscribed at extraction or never paid.

we can see that Plan without payment: 0. Paid without plan: 0. End date without start date: 0. Paid without trial: 0 (every paying user went through a trial).

for that No rows were removed or changed in this step.

## Saving our cleaned Data

In [52]:
user.to_csv("../data/users_cleaned.csv")
mark.to_csv("../data/marketing_spend_cleaned.csv")
